## Multiple Lineare Regression - Fallstudie zum Wohnungswesen

Problemstellung:

Nehmen wir an, ein Immobilienunternehmen verfügt über die Daten der Immobilienpreise in einer Stadt. 
Das Unternehmen möchte den Verkaufspreis der Immobilien auf der Grundlage wichtiger Faktoren wie Fläche, Schlafzimmer, Parkplätze usw. optimieren.
Das Unternehmen möchte im Wesentlichen:

1. Ermittlung der Variablen, die sich auf die Hauspreise auswirken, z. B. Fläche, Anzahl der Zimmer, Bäder usw.
2. Erstellung eines linearen Modells, das die Hauspreise quantitativ mit Variablen wie der Anzahl der Zimmer, der Fläche, der Anzahl der Bäder usw. in Beziehung setzt.
3. Ermittlung der Genauigkeit des Modells, d. h. wie gut diese Variablen die Hauspreise vorhersagen.

Deshalb ist die Interpretation so wichtig!

Die in dem Modell betrachtenten Variablen sind:
- price: ist der Verkaufspreis eines Hauses in Rs.
- area: gibt die Gesamtgröße einer Immobilie in Quadratfuss an.
- bedrooms: steht für die Anzahl der Schlafzimmer.
- bathrooms: gibt die Anzahl der Bäder an.
- stories: gibt die Anzahl der Stockwerke ohne Keller an.
- mainroad: = 1, wenn das Haus an einer Hauptstraße liegt.
- guestroom: = 1, wenn das Haus ein separates Zimmer für Gäste hat.
- basement: zeigt an, ob das Haus einen Keller hat.
- hotwaterheating: = 1, wenn das Haus über Warmwasser-Heizung verfügt.
- airconditioning: = 1, wenn es eine zentrale Klimaanlage gibt.
- parking: shoes die Anzahl der Autos, die geparkt werden können.
- prefarea: yes, wenn das Haus im bevorzugten Viertel der Stadt liegt.
- furnishingstatus: kann den Wert furnished (möbliert), unfurnished (unmöbliert) oder semi-furnished (teilmöbliert) annehmen.

In [ ]:
# Warnungen unterdrücken
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Bibliotheken installieren (falls Sie dies noch nicht getan haben)
# !pip install pandas
# !pip install numpy
# !pip install seaborn
# !pip install matplotlib
# !pip install sklearn
# !pip install statsmodels

In [ ]:
# Bibliotheken importieren
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import statsmodels.api as sm

#### Laden Sie die Daten:

In [ ]:
# Datei einlesen
housing_df = pd.read_csv('Housing.csv')
housing_df.head()

In [ ]:
# Form des dataframes
housing_df.shape

In [ ]:
# Informationen
housing_df.info()

In [ ]:
# Deskriptive Statistik
housing_df.describe(percentiles = [0.10,0.25,0.50,0.75,0.90,0.99])

#### Visualizing the data:

In [ ]:
# Paardarstellung der numerischen Variablen
sns.pairplot(data = housing_df)

In [ ]:
# Paarplot der kategorialen Variablen
plt.figure(figsize = (20,8))

plt.subplot(2,3,1)
sns.boxplot(x ='mainroad', y = 'price', data = housing_df)
plt.subplot(2,3,2)
sns.boxplot(x ='guestroom', y = 'price', data = housing_df)
plt.subplot(2,3,3)
sns.boxplot(x ='basement', y = 'price', data = housing_df)
plt.subplot(2,3,4)
sns.boxplot(x ='hotwaterheating', y = 'price', data = housing_df)
plt.subplot(2,3,5)
sns.boxplot(x ='airconditioning', y = 'price', data = housing_df)
plt.subplot(2,3,6)
sns.boxplot(x ='prefarea', y = 'price', data = housing_df)

In [ ]:
sns.boxplot(x ='furnishingstatus', y = 'price', data = housing_df)

In [ ]:
plt.figure(figsize = (10,5))
sns.boxplot(x ='furnishingstatus', y = 'price', hue = 'airconditioning', data = housing_df)

### Was beobachten Sie?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

### Aufbereitung der Daten(Regression):

In [ ]:
# Ja in 1 und Nein in 0 umwandeln
variable_list = ['mainroad','guestroom','basement','hotwaterheating','airconditioning', 'prefarea']

def binary_map(x):
    return x.map({'yes':1 , 'no':0})

housing_df[variable_list] = housing_df[variable_list].apply(binary_map)

In [ ]:
housing_df.head()

#### Dummy Variablen:

In [ ]:
# Erstellen Sie eine Dummy-Variable für den Einrichtungsstatus ('furnishingstatus') mit 3 Werten
status = pd.get_dummies(housing_df['furnishingstatus'],  dtype="int")
status.head()

In [ ]:
# Dummy-Variablen zum vorhandenen Datensatzu hinzufügen
housing_df = pd.concat([housing_df,status], axis =1)

In [ ]:
housing_df.head()

In [ ]:
# Entfernen der Variable "furnishing status", die in Dummy-Variables umgewandelt wurde
housing_df.drop(['furnishingstatus'], axis =1, inplace = True)

In [ ]:
housing_df.head()

In [ ]:
# Korrelationen
sns.heatmap(housing_df.corr(),annot = True)

#### Aufteilung der Daten in Test-Train-Split

In [ ]:
df_train, df_test = train_test_split(housing_df, train_size = 0.7, test_size = 0.3, random_state = 100)

In [ ]:
df_train.shape

#### Skalieren Sie die Merkmale neu:

In [ ]:
# Skalierung zwischen 0 und 1. 
scaler = MinMaxScaler()

#applying the scaler only to below variables
num_var= ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']

df_train[num_var] = scaler.fit_transform(df_train[num_var])

In [ ]:
df_train.head()

In [ ]:
df_train.describe()

Alle Werte liegen im Bereich zwischen 0 und 1.

In [ ]:
# Überprüfen Sie die Korrelation der "Train" Daten
plt.figure(figsize = (10,8))
sns.heatmap(df_train.corr(),annot = True, cmap = 'YlGnBu')

### Welche Variablen weisen eine hohe Korrelation auf?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

#### Aufteilung von X und Y für die Modellbildung:

In [ ]:
# Die Funktion pop entfernt die Zielvariable aus dem DataFrame und gibt sie zurück
y_train = df_train.pop('price')
X_train = df_train

### Erstellung eines linearen Modells:


##### Folgen wir einem Bottom-up-Ansatz, d. h. wir beginnen mit der Erstellung des Modells mit nur einer Variablen:

In [ ]:
# Wir nehmen die Fläche als erklärende Variable ('area')
X_train_sm = sm.add_constant(X_train[['area']])

lr_1 = sm.OLS(y_train, X_train_sm).fit()

In [ ]:
lr_1.params

In [ ]:
print(lr_1.summary())

### Welche Erklärungskraft hat dieses Modell?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

In [ ]:
# Fügen wir nun eine Variable "Badezimmer" hinzu ('area' und 'bathrooms')
X_train_sm = sm.add_constant(X_train[['area', 'bathrooms']])

lr_2 = sm.OLS(y_train, X_train_sm).fit()

In [ ]:
lr_2.params

In [ ]:
print(lr_2.summary())

### Was beobachten Sie ? <span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

In [ ]:
# Fügen Sie eine weitere Variable (Schlafzimmer) hinzu und überprüfen sie:

# area ,bathrooms and bedrooms
X_train_sm = sm.add_constant(X_train[['area','bedrooms','bathrooms']])

lr_3 = sm.OLS(y_train, X_train_sm).fit()

In [ ]:
lr_3.params

In [ ]:
print(lr_3.summary())

### Was beobachten Sie ? <span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

### Gehen wir den umgekehrten Weg: Wir erstellen das Modell, indem wir alle Variablen in das Modell aufnehmen und diejenigen, die nicht signifikant sind, weglassen:

In [ ]:
X_train.columns

In [ ]:
X_train_sm = sm.add_constant(X_train)

lr_4 = sm.OLS(y_train, X_train_sm).fit()

In [ ]:
lr_4.params

In [ ]:
print(lr_4.summary())

### Welche Variablen würden Sie entfernen? Begründen Sie Ihre Antwort!<span style="float: right;">(0.5 Punkt)</span>

<p>Ihre Antwort</p>

In [ ]:
# Weglassen der Variable 
# Ersetzen Sie die Variable area zwischen den Anführungszeichen mit der von Ihnen ausgewählten Variable
X = X_train.drop('area', axis =1)

In [ ]:
X_sm = sm.add_constant(X)
lr_5 = sm.OLS(y_train, X_sm).fit()

In [ ]:
print(lr_5.summary())

### Welche Variable wird nun statistisch nicht signifikant?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

In [ ]:
# Weglassen der Variable 
# Ersetzen Sie die Variable area zwischen den Anführungszeichen mit der von Ihnen ausgewählten Variable
X = X.drop('parking', axis =1)

In [ ]:
X_sm = sm.add_constant(X)
lr_6 = sm.OLS(y_train, X_sm).fit()

In [ ]:
print(lr_6.summary())

### Was beobachten Sie?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>

### Residualanalyse der Train Daten

In [ ]:
y_train_pred = lr_6.predict(X_sm)

In [ ]:
residual = y_train- y_train_pred

In [ ]:
sns.histplot(residual, bins=20, kde=True, edgecolor='lightblue')
# Add labels and title
plt.title("Distribution of Residuals")
plt.xlabel("Residuals")
plt.ylabel("Frequency")


### Was können Sie über die Verteilung der Residuen (Fehlerterm) sagen?<span style="float: right;">(0.5 Punkt)</span>

<p>Ihre Antwort</p>

#### Prognosen anhand des endgültigen Modells:

In [ ]:
# Diese Variablen haben wir in den Train-Daten skaliert, also sollten wir 
# die gleichen Variablen auch in den Testdaten skalieren. 
num_var= ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']

df_test[num_var] = scaler.transform(df_test[num_var])

In [ ]:
df_test.describe()

In [ ]:
y_test = df_test.pop('price')
X_test = df_test

In [ ]:
X_test_sm = sm.add_constant(X_test)

In [ ]:
X_test_sm = X_test_sm.drop(['semi-furnished','bedrooms'], axis =1)

In [ ]:
y_pred = lr_6.predict(X_test_sm)

#### Modellbewertung:

In [ ]:
plt.scatter(y_test, y_pred)

In [ ]:
lr_6.summary()

### Wie lautet die Gleichung des Regressionsmodells?<span style="float: right;">(0.25 Punkt)</span>

<p>Ihre Antwort</p>